In [1]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install unsloth

In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

models = {
  "llama": "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
  "mistral": "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
  "gemma": "unsloth/gemma-3-4b-it",
  "mistral-nemo": "unsloth/Mistral-Nemo-Base-2407"
}


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = models["mistral-nemo"],
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

<ipython-input-3-72c1c6c28b1d>:1: UserWarning: WARNING: Unsloth should be imported before trl, transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Mistral patching. Transformers: 4.50.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.31G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/162 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/177k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.3.19 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [6]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "mistral", # Supports zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, unsloth, qwen-2.5
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
    map_eos_token = True, # Maps <|im_end|> to </s> instead
)

def formatting_prompts_func(examples):
    print(examples)
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("json", data_files="/content/swrl_dataset_v2.json", split='train')
dataset = dataset.map(formatting_prompts_func, batched = True,)


Map:   0%|          | 0/95 [00:00<?, ? examples/s]

{'conversations': [[{'content': 'Act as an ontology engineer. Use the given ontology content to translate the natural language rule into a SWRL rule. Give only the final SWRL rule.', 'role': 'system'}, {'content': 'PART OF ONTOLOGY\n\nisParentOf (domain: Person, range: Person)\nhasChild (domain: Person, range: Person)\n\nEND OF ONTOLOGY\n\nThe rule in natural language is the following: If a person is a parent of another person, then the first person has a child who is the second person.', 'role': 'user'}, {'content': 'Person(?p1) ∧ Person(?p2) ∧ isParentOf(?p1, ?p2) -> hasChild(?p1, ?p2)', 'role': 'assistant'}], [{'content': 'Act as an ontology engineer. Use the given ontology content to translate the natural language rule into a SWRL rule. Give only the final SWRL rule.', 'role': 'system'}, {'content': 'PART OF ONTOLOGY\n\nhasParent(domain: Person, range: Person)\ndifferentFrom(domain: Person, range: Person)\nisSiblingOf(domain: Person, range: Person)\n\nEND OF ONTOLOGY\n\nThe rule in

In [7]:
dataset[5]["conversations"]

[{'content': 'Act as an ontology engineer. Use the given ontology content to translate the natural language rule into a SWRL rule. Give only the final SWRL rule.',
  'role': 'system'},
 {'content': 'PART OF ONTOLOGY\n\nisEnrolledIn(domain: Student, range: Course)\nhasParticipant(domain: Course, range: Student)\n\nEND OF ONTOLOGY\n\nThe rule in natural language is the following: If a student is enrolled in a course, then the course has that student as a participant.',
  'role': 'user'},
 {'content': 'Student(?student) ∧ Course(?course) ∧ isEnrolledIn(?student, ?course) -> hasParticipant(?course, ?student)',
  'role': 'assistant'}]

In [8]:
print(dataset[5]["text"])

<s>[INST] Act as an ontology engineer. Use the given ontology content to translate the natural language rule into a SWRL rule. Give only the final SWRL rule. PART OF ONTOLOGY

isEnrolledIn(domain: Student, range: Course)
hasParticipant(domain: Course, range: Student)

END OF ONTOLOGY

The rule in natural language is the following: If a student is enrolled in a course, then the course has that student as a participant. [/INST]Student(?student) ∧ Course(?course) ∧ isEnrolledIn(?student, ?course) -> hasParticipant(?course, ?student)</s>


In [12]:
from unsloth import apply_chat_template
chat_template = """{SYSTEM}
USER: {INPUT}
ASSISTANT: {OUTPUT}"""


dataset = apply_chat_template(
    dataset,
    tokenizer = tokenizer,
    chat_template = chat_template
)


Unsloth: We automatically added an EOS token to stop endless generations.


Map:   0%|          | 0/95 [00:00<?, ? examples/s]

In [10]:
def apply_chat_template(examples):
    texts = tokenizer.apply_chat_template(examples["conversations"])
    return { "text" : texts }
pass
dataset = dataset.map(apply_chat_template, batched = True)

Map:   0%|          | 0/95 [00:00<?, ? examples/s]

In [29]:
dataset[5]

{'conversations': [{'content': 'Act as an ontology engineer. Use the given ontology content to translate the natural language rule into a SWRL rule. Give only the final SWRL rule.',
   'role': 'system'},
  {'content': 'PART OF ONTOLOGY\n\nisEnrolledIn(domain: Student, range: Course)\nhasParticipant(domain: Course, range: Student)\n\nEND OF ONTOLOGY\n\nThe rule in natural language is the following: If a student is enrolled in a course, then the course has that student as a participant.',
   'role': 'user'},
  {'content': 'Student(?student) ∧ Course(?course) ∧ isEnrolledIn(?student, ?course) -> hasParticipant(?course, ?student)',
   'role': 'assistant'}],
 'text': '<s>Act as an ontology engineer. Use the given ontology content to translate the natural language rule into a SWRL rule. Give only the final SWRL rule.\nUSER: PART OF ONTOLOGY\n\nisEnrolledIn(domain: Student, range: Course)\nhasParticipant(domain: Course, range: Student)\n\nEND OF ONTOLOGY\n\nThe rule in natural language is the

In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 10, # Set num_train_epochs = 1 for full training runs
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/95 [00:00<?, ? examples/s]

In [15]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 95 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 57,016,320/4,000,000,000 (1.43% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.526000
2,1.388000
3,1.477700
4,1.459800
5,1.289700
6,1.035700
7,0.836100
8,0.688500
9,0.544200
10,0.559600


In [16]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [18]:
# Save to 8bit Q8_0
if True: model.save_pretrained_gguf("model", tokenizer,)
# if True: model.save_pretrained_gguf("model", quantization_type="q8_0")

# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 8.3G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 60.84 out of 83.48 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 40/40 [00:00<00:00, 48.65it/s]


Unsloth: Saving tokenizer... Done.
Done.


Unsloth: Converting mistral model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at model into q8_0 GGUF format.
The output location will be /content/model/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00005.safetensors'
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> Q8_0, shape = {

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Conversion completed! Output location: /content/model/unsloth.Q8_0.gguf
Unsloth: Saved Ollama Modelfile to model/Modelfile


In [ ]:
import subprocess
subprocess.Popen(["ollama", "serve"])
import time
time.sleep(3) # Wait for a few seconds for Ollama to load!

In [ ]:

tokenizer._ollama_modelfile = "/content/model/unsloth.Q8_0.gguf"
print(tokenizer._ollama_modelfile)

/content/model/unsloth.Q8_0.gguf


In [ ]:
!ollama create unsloth_model -f ./model/Modelfile

gathering model components 
Error: no Modelfile or safetensors files found


In [ ]:
!pkill ollama  # Stops the current Ollama process
!ollama serve  # Starts Ollama again

2025/03/12 20:10:32 routes.go:1225: INFO server config env="map[CUDA_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:2048 OLLAMA_DEBUG:false OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_INTEL_GPU:false OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NUM_PARALLEL:0 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_SCHED_SPREAD:false ROCR_VISIBLE_DEVICES: http_proxy: https_proxy: